# Proposed pipeline (v2 — label-mapping fix)

Same TF-IDF + Chi2/MI feature selection + GridSearchCV pipeline as
`proposed_improvements.ipynb`, with the label-mapping bug fixed (I-1) and a corrected
evaluation: GridSearchCV now optimizes macro-F1 instead of positive-class-only F1, and
results report macro-F1, per-class precision/recall/F1, and a confusion matrix for every
model (I-2). `class_weight='balanced'` is added as a tunable option for LR/SVM (I-5).
Results are merged with the baseline + Dummy results from `baseline_pipeline_v2.ipynb`
into one comparable table (I-4). SHAP explainability is deferred to a later pass once the
corrected numbers are the ones being explained (see ISSUE_PLAN.md Phase 5).

**2026-08-24 update (CV-fold feature-selection leakage fix):** the Chi2/MI selectors
were previously fit once on the full training set (using `y_train`) *before*
`GridSearchCV`'s internal 5-fold CV ran -- a real, if narrow-impact, form of leakage
(selection saw labels of rows that later served as held-out CV folds during
hyperparameter search; this does not affect the test-set numbers, since selection was
never fit on valid/test, but it could bias which hyperparameters GridSearchCV picked as
"best"). Selection is now wrapped in an `sklearn.Pipeline` with the classifier so it is
refit inside every CV fold, matching the leak-free pattern `ablation_v1.ipynb` already
used.

In [1]:
import re
import warnings

import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

from liar_utils import RANDOM_STATE, evaluate_full, load_and_label, print_report

Load data with the corrected label mapping

In [2]:
train = load_and_label("train.csv")
valid = load_and_label("valid.csv")
test = load_and_label("test.csv")

balance = train["Label"].value_counts(normalize=True).rename({0: "fake", 1: "real"})
print("Train class balance:\n", balance)
assert 0.35 < balance["fake"] < 0.5, "Fake-class share outside the expected ~44% range -- check label mapping"

Train class balance:
 Label
real    0.561719
fake    0.438281
Name: proportion, dtype: float64


Text preprocessing (stopword removal + stemming)

In [3]:
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()


def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z]", " ", text)
    words = text.split()
    words = [stemmer.stem(w) for w in words if w not in stop_words]
    return " ".join(words)


train["clean_text"] = train["Statement"].apply(preprocess)
valid["clean_text"] = valid["Statement"].apply(preprocess)
test["clean_text"] = test["Statement"].apply(preprocess)

TF-IDF feature extraction

In [4]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2, max_df=0.9)

X_train_tfidf = tfidf.fit_transform(train["clean_text"])
X_valid_tfidf = tfidf.transform(valid["clean_text"])
X_test_tfidf = tfidf.transform(test["clean_text"])

y_train = train["Label"]
y_valid = valid["Label"]
y_test = test["Label"]

print("TF-IDF shape:", X_train_tfidf.shape)

TF-IDF shape: (10240, 10000)


Chi-Square feature selection -- selector *factory* only. The selector itself is fit
inside each GridSearchCV/CV fold (see the Pipeline in the next code section), not here,
so it never sees labels from rows that later serve as a held-out CV fold.

In [5]:
k_features = 3000


def make_chi2_selector():
    return SelectKBest(score_func=chi2, k=k_features)


print(f"Chi-square selector factory ready (k={k_features}); fit happens per-CV-fold below.")

Chi-square selector factory ready (k=3000); fit happens per-CV-fold below.


Mutual Information feature selection -- selector factory only, same reasoning as the
Chi-square cell above: fit happens inside each CV fold, not on the full training set
beforehand.

In [6]:
def _mutual_info_score_func(X, y):
    # sklearn requires discrete_features=True/"auto" for sparse input (a dense
    # continuous-feature estimator isn't supported here), which routes through an
    # internal discrete-discrete MI helper that emits one benign UserWarning per
    # feature ("Clustering metrics expects discrete values..."). At 10,000 features x
    # every CV fold x every hyperparameter combination, that floods captured notebook
    # output to hundreds of MB. Suppressing it here (inside the function actually
    # executed by each GridSearchCV worker) is reliable across process boundaries in a
    # way that a notebook-level `warnings.filterwarnings` call is not; it changes no
    # computation, only what gets printed.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return mutual_info_classif(X, y, random_state=RANDOM_STATE)


def make_mi_selector():
    return SelectKBest(score_func=_mutual_info_score_func, k=k_features)


print(f"Mutual Information selector factory ready (k={k_features}); fit happens per-CV-fold below.")

Mutual Information selector factory ready (k=3000); fit happens per-CV-fold below.


GridSearch + evaluation -- macro-F1 is now the tuning objective (I-2). Feature selection
(chi2/MI) is wrapped in the same `sklearn.Pipeline` as the classifier and fit inside each
CV fold, so `GridSearchCV`'s internal cross-validation never sees a selector fit on labels
from its own held-out fold (matches the leak-free pattern `ablation_v1.ipynb` already
used for its Chi2 stage). Chi-square is cheap enough per fold that this refit adds
negligible runtime; Mutual Information is more expensive per fold but still tractable at
this dataset size without additional caching.

In [7]:
def train_and_evaluate(model, param_grid, selector_factory, X_train, y_train, X_valid, y_valid, X_test, y_test):
    pipe = Pipeline([("select", selector_factory()), ("clf", model)])
    prefixed_grid = {f"clf__{k}": v for k, v in param_grid.items()}
    grid = GridSearchCV(pipe, prefixed_grid, cv=5, scoring="f1_macro", n_jobs=-1)
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_

    valid_metrics = evaluate_full(y_valid, best_model.predict(X_valid))
    test_metrics = evaluate_full(y_test, best_model.predict(X_test))

    return best_model, grid.best_params_, valid_metrics, test_metrics

Model + param-grid definitions (class_weight='balanced' added to LR/SVM per I-5)

In [8]:
def make_models():
    return [
        (
            "Logistic Regression",
            LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
            {"C": [0.1, 1, 10], "solver": ["liblinear"], "class_weight": [None, "balanced"]},
        ),
        (
            "SVM",
            LinearSVC(random_state=RANDOM_STATE),
            {"C": [0.1, 1, 10], "class_weight": [None, "balanced"]},
        ),
        ("Naive Bayes", MultinomialNB(), {"alpha": [0.1, 0.5, 1.0]}),
        (
            "Random Forest",
            RandomForestClassifier(random_state=RANDOM_STATE),
            {
                "n_estimators": [100, 200],
                "max_depth": [None, 10],
                "min_samples_split": [2, 5],
            },
        ),
        (
            "XGBoost",
            XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE),
            {
                "n_estimators": [100, 200],
                "max_depth": [3, 6],
                "learning_rate": [0.01, 0.1],
            },
        ),
    ]

Run experiments (Chi² features)

In [9]:
results_chi2 = {}
best_models_chi2 = {}

for name, model, params in make_models():
    print(f"\nTraining {name} with Chi-square features (selected inside each CV fold)...")
    best_model, best_params, valid_metrics, test_metrics = train_and_evaluate(
        model, params, make_chi2_selector, X_train_tfidf, y_train, X_valid_tfidf, y_valid, X_test_tfidf, y_test
    )
    print("Best params:", best_params)
    print_report(name, y_test, best_model.predict(X_test_tfidf))
    results_chi2[name] = {"best_params": best_params, "valid": valid_metrics, "test": test_metrics}
    best_models_chi2[name] = best_model


Training Logistic Regression with Chi-square features (selected inside each CV fold)...


Best params: {'clf__C': 0.1, 'clf__class_weight': 'balanced', 'clf__solver': 'liblinear'}

Logistic Regression
[[337 216]
 [274 440]]
              precision    recall  f1-score   support

        fake      0.552     0.609     0.579       553
        real      0.671     0.616     0.642       714

    accuracy                          0.613      1267
   macro avg      0.611     0.613     0.611      1267
weighted avg      0.619     0.613     0.615      1267


Training SVM with Chi-square features (selected inside each CV fold)...


Best params: {'clf__C': 0.1, 'clf__class_weight': 'balanced'}

SVM
[[315 238]
 [271 443]]
              precision    recall  f1-score   support

        fake      0.538     0.570     0.553       553
        real      0.651     0.620     0.635       714

    accuracy                          0.598      1267
   macro avg      0.594     0.595     0.594      1267
weighted avg      0.601     0.598     0.599      1267


Training Naive Bayes with Chi-square features (selected inside each CV fold)...
Best params: {'clf__alpha': 0.5}

Naive Bayes
[[203 350]
 [153 561]]
              precision    recall  f1-score   support

        fake      0.570     0.367     0.447       553
        real      0.616     0.786     0.690       714

    accuracy                          0.603      1267
   macro avg      0.593     0.576     0.569      1267
weighted avg      0.596     0.603     0.584      1267


Training Random Forest with Chi-square features (selected inside each CV fold)...


Best params: {'clf__max_depth': None, 'clf__min_samples_split': 5, 'clf__n_estimators': 100}

Random Forest
[[264 289]
 [209 505]]
              precision    recall  f1-score   support

        fake      0.558     0.477     0.515       553
        real      0.636     0.707     0.670       714

    accuracy                          0.607      1267
   macro avg      0.597     0.592     0.592      1267
weighted avg      0.602     0.607     0.602      1267


Training XGBoost with Chi-square features (selected inside each CV fold)...


Best params: {'clf__learning_rate': 0.1, 'clf__max_depth': 6, 'clf__n_estimators': 200}

XGBoost
[[180 373]
 [129 585]]
              precision    recall  f1-score   support

        fake      0.583     0.325     0.418       553
        real      0.611     0.819     0.700       714

    accuracy                          0.604      1267
   macro avg      0.597     0.572     0.559      1267
weighted avg      0.598     0.604     0.577      1267



Run experiments (MI features)

In [10]:
results_mi = {}
best_models_mi = {}

for name, model, params in make_models():
    print(f"\nTraining {name} with MI features (selected inside each CV fold)...")
    best_model, best_params, valid_metrics, test_metrics = train_and_evaluate(
        model, params, make_mi_selector, X_train_tfidf, y_train, X_valid_tfidf, y_valid, X_test_tfidf, y_test
    )
    print("Best params:", best_params)
    print_report(name, y_test, best_model.predict(X_test_tfidf))
    results_mi[name] = {"best_params": best_params, "valid": valid_metrics, "test": test_metrics}
    best_models_mi[name] = best_model


Training Logistic Regression with MI features (selected inside each CV fold)...


Best params: {'clf__C': 0.1, 'clf__class_weight': 'balanced', 'clf__solver': 'liblinear'}

Logistic Regression
[[340 213]
 [266 448]]
              precision    recall  f1-score   support

        fake      0.561     0.615     0.587       553
        real      0.678     0.627     0.652       714

    accuracy                          0.622      1267
   macro avg      0.619     0.621     0.619      1267
weighted avg      0.627     0.622     0.623      1267


Training SVM with MI features (selected inside each CV fold)...


Best params: {'clf__C': 0.1, 'clf__class_weight': 'balanced'}

SVM
[[307 246]
 [268 446]]
              precision    recall  f1-score   support

        fake      0.534     0.555     0.544       553
        real      0.645     0.625     0.634       714

    accuracy                          0.594      1267
   macro avg      0.589     0.590     0.589      1267
weighted avg      0.596     0.594     0.595      1267


Training Naive Bayes with MI features (selected inside each CV fold)...


Best params: {'clf__alpha': 0.1}

Naive Bayes
[[221 332]
 [173 541]]
              precision    recall  f1-score   support

        fake      0.561     0.400     0.467       553
        real      0.620     0.758     0.682       714

    accuracy                          0.601      1267
   macro avg      0.590     0.579     0.574      1267
weighted avg      0.594     0.601     0.588      1267


Training Random Forest with MI features (selected inside each CV fold)...


Best params: {'clf__max_depth': None, 'clf__min_samples_split': 2, 'clf__n_estimators': 200}

Random Forest
[[254 299]
 [186 528]]
              precision    recall  f1-score   support

        fake      0.577     0.459     0.512       553
        real      0.638     0.739     0.685       714

    accuracy                          0.617      1267
   macro avg      0.608     0.599     0.598      1267
weighted avg      0.612     0.617     0.609      1267


Training XGBoost with MI features (selected inside each CV fold)...


Best params: {'clf__learning_rate': 0.1, 'clf__max_depth': 6, 'clf__n_estimators': 200}

XGBoost
[[205 348]
 [139 575]]
              precision    recall  f1-score   support

        fake      0.596     0.371     0.457       553
        real      0.623     0.805     0.703       714

    accuracy                          0.616      1267
   macro avg      0.609     0.588     0.580      1267
weighted avg      0.611     0.616     0.595      1267



Results to table, merged with the v2 baseline + Dummy results (I-4 exit criterion)

In [11]:
def results_to_dataframe(results_dict, method_name):
    rows = []
    for model_name, data in results_dict.items():
        valid_m = data["valid"]
        test_m = data["test"]
        rows.append(
            {
                "Pipeline": "Proposed",
                "Method": method_name,
                "Model": model_name,
                "Valid Accuracy": valid_m["accuracy"],
                "Valid Macro-F1": valid_m["macro_f1"],
                "Valid Fake F1": valid_m["fake_f1"],
                "Test Accuracy": test_m["accuracy"],
                "Test Macro-F1": test_m["macro_f1"],
                "Test Fake Precision": test_m["fake_precision"],
                "Test Fake Recall": test_m["fake_recall"],
                "Test Fake F1": test_m["fake_f1"],
                "Test Real F1": test_m["real_f1"],
                "Test Confusion Matrix": test_m["confusion_matrix"],
                "Best Params": data["best_params"],
            }
        )
    return pd.DataFrame(rows)


df_chi2 = results_to_dataframe(results_chi2, "Chi-square")
df_mi = results_to_dataframe(results_mi, "Mutual Information")
proposed_results = pd.concat([df_chi2, df_mi], ignore_index=True)

baseline_results = pd.read_csv("baseline_results_v2.csv")

final_results = pd.concat([baseline_results, proposed_results], ignore_index=True)
final_results = final_results.sort_values("Test Macro-F1", ascending=False)
final_results

,Pipeline,Method,Model,Valid Accuracy,Valid Macro-F1,Valid Fake F1,Test Accuracy,Test Macro-F1,Test Fake Precision,Test Fake Recall,Test Fake F1,Test Real F1,Test Confusion Matrix,Best Params
13,Proposed,Mutual Information,Logistic Regression,0.626947,0.626929,0.624314,0.621942,0.619175,0.561056,0.614828,0.586713,0.651636,"[[340, 213], [266, 448]]","{'clf__C': 0.1, 'clf__class_weight': 'balanced..."
8,Proposed,Chi-square,Logistic Regression,0.619938,0.619937,0.619345,0.613260,0.610687,0.551555,0.609403,0.579038,0.642336,"[[337, 216], [274, 440]]","{'clf__C': 0.1, 'clf__class_weight': 'balanced..."
3,Baseline,TF-IDF only,Logistic Regression,0.612928,0.603981,0.544455,0.621942,0.600735,0.587678,0.448463,0.508718,0.692752,"[[248, 305], [174, 540]]",NaN
16,Proposed,Mutual Information,Random Forest,0.625389,0.616359,0.557498,0.617206,0.598425,0.577273,0.459313,0.511581,0.685269,"[[254, 299], [186, 528]]","{'clf__max_depth': None, 'clf__min_samples_spl..."
4,Baseline,TF-IDF only,Logistic Regression (balanced),0.605919,0.605642,0.595200,0.602210,0.597778,0.542169,0.569620,0.555556,0.640000,"[[315, 238], [266, 448]]",NaN
9,Proposed,Chi-square,SVM,0.619159,0.619120,0.615264,0.598264,0.594121,0.537543,0.569620,0.553117,0.635125,"[[315, 238], [271, 443]]","{'clf__C': 0.1, 'clf__class_weight': 'balanced'}"
11,Proposed,Chi-square,Random Forest,0.602804,0.594841,0.538043,0.606946,0.592191,0.558140,0.477396,0.514620,0.669761,"[[264, 289], [209, 505]]","{'clf__max_depth': None, 'clf__min_samples_spl..."
5,Baseline,TF-IDF only,SVM,0.593458,0.591173,0.560606,0.602210,0.591301,0.548323,0.502712,0.524528,0.658073,"[[278, 275], [229, 485]]",NaN
14,Proposed,Mutual Information,SVM,0.609813,0.609555,0.599520,0.594317,0.589375,0.533913,0.555154,0.544326,0.634424,"[[307, 246], [268, 446]]","{'clf__C': 0.1, 'clf__class_weight': 'balanced'}"
6,Baseline,TF-IDF only,SVM (balanced),0.591121,0.590851,0.580336,0.591160,0.586878,0.529915,0.560579,0.544815,0.628940,"[[310, 243], [275, 439]]",NaN


In [12]:
final_results.to_csv("model_comparison_results_v2.csv", index=False)